# chess-gnn: supervised training with Stockfish value targets

This notebook builds a sidecar dataset of Stockfish evaluations for sampled PGN positions, then trains the GNN with the played move as the policy target and the engine evaluation as the value target.

Use this when the game-result value head is too flat for MCTS. The original `train_sl.ipynb` remains unchanged; this is a separate workflow for MCTS-ready value supervision.

In [ ]:
# --- environment setup ---------------------------------------------------
import sys, subprocess, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules
print("colab:", IN_COLAB)

REPO_URL = "https://github.com/jordanshivers/chess-gnn"
REPO_DIR = pathlib.Path("/content/chess") if IN_COLAB else pathlib.Path.cwd().parent

if IN_COLAB:
    if REPO_URL and not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    subprocess.check_call([
        "pip", "install", "-q",
        "torch", "torch-geometric", "python-chess", "zstandard", "tqdm",
    ])
    subprocess.check_call(["pip", "install", "-q", "-e", str(REPO_DIR)])
    if shutil.which("stockfish") is None and not pathlib.Path("/usr/games/stockfish").exists():
        subprocess.check_call(["apt-get", "update", "-qq"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", "stockfish"])

src_path = str((REPO_DIR / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
print("torch:", torch.__version__)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE)

In [ ]:
# --- data and engine paths -----------------------------------------------
DATA_DIR = REPO_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PGN_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2019-01.pgn.zst"
PGN_PATH = DATA_DIR / pathlib.Path(PGN_URL).name
if not PGN_PATH.exists():
    subprocess.check_call(["curl", "-L", "--progress-bar", "-o", str(PGN_PATH), PGN_URL])
print("pgn:", PGN_PATH, f"{PGN_PATH.stat().st_size / 1e6:.1f} MB")

STOCKFISH = shutil.which("stockfish") or "/usr/games/stockfish"
assert pathlib.Path(STOCKFISH).exists() or shutil.which(STOCKFISH), f"No Stockfish at {STOCKFISH}"
print("stockfish:", STOCKFISH)

VALUE_DATA = DATA_DIR / "stockfish_values_2019-01.pt"
MIN_ELO = 1800

In [ ]:
# --- generate Stockfish value labels -------------------------------------
# Start small to validate the workflow. Increase POSITIONS to 100k-500k for a real run.
from chess_gnn.make_value_dataset import generate

POSITIONS = 500_000

if not VALUE_DATA.exists():
    generate(
        pgn_paths=[PGN_PATH],
        out=VALUE_DATA,
        stockfish=STOCKFISH,
        positions=POSITIONS,
        depth=8,
        min_elo=MIN_ELO,
        min_ply=8,
        sample_every=4,
        max_positions_per_game=8,
        cp_scale=400.0,
    )
else:
    print("using existing", VALUE_DATA)

In [ ]:
# --- inspect labels and define held-out eval -----------------------------
import torch.nn.functional as F
from torch_geometric.data import Batch

from chess_gnn.dataset import EngineValueDataset
from chess_gnn.model import load_model, masked_log_softmax
from chess_gnn.train_sl import topk_accuracy

payload = torch.load(VALUE_DATA, map_location="cpu", weights_only=False)
records = payload["records"]
HELDOUT_SIZE = min(512, max(1, len(records) // 10))
TRAIN_VALUE_DATA = VALUE_DATA.with_name(VALUE_DATA.stem + "_train.pt")
train_payload = dict(payload)
train_payload["records"] = records[:-HELDOUT_SIZE]
torch.save(train_payload, TRAIN_VALUE_DATA)

train_value_ds = EngineValueDataset(TRAIN_VALUE_DATA, value_blend=1.0)
full_value_ds = EngineValueDataset(VALUE_DATA, value_blend=1.0)
print("engine-value records:", len(records), "train:", len(train_value_ds), "heldout:", HELDOUT_SIZE)
sample = train_value_ds[0]
print("sample value_target:", float(sample.value_target), "legal moves:", int(sample.legal_mask.sum()))

heldout_samples = [full_value_ds[i] for i in range(len(full_value_ds) - HELDOUT_SIZE, len(full_value_ds))]

def evaluate_sl_checkpoint(ckpt_path, samples=heldout_samples, device=DEVICE, batch_size=64):
    model = load_model(ckpt_path, device=device)
    model.eval()
    totals = {"n": 0, "policy": 0.0, "value": 0.0, "top1": 0.0, "top5": 0.0, "value_sum": 0.0, "target_sum": 0.0}
    with torch.no_grad():
        for start in range(0, len(samples), batch_size):
            batch = Batch.from_data_list(samples[start : start + batch_size]).to(device)
            policy, _, value = model(batch)
            mask = batch.legal_mask.view(-1, policy.size(-1))
            log_probs = masked_log_softmax(policy, mask)
            value_target = batch.value_target.view_as(value).to(dtype=value.dtype)
            n = batch.y.size(0)
            totals["n"] += n
            totals["policy"] += F.nll_loss(log_probs, batch.y, reduction="sum").item()
            totals["value"] += F.mse_loss(value, value_target, reduction="sum").item()
            totals["top1"] += topk_accuracy(log_probs, batch.y, 1) * n
            totals["top5"] += topk_accuracy(log_probs, batch.y, 5) * n
            totals["value_sum"] += float(value.sum().item())
            totals["target_sum"] += float(value_target.sum().item())
    n = max(totals["n"], 1)
    print(
        f"eval {pathlib.Path(ckpt_path).name}: "
        f"policy {totals['policy']/n:.4f} | v {totals['value']/n:.4f} | "
        f"top1 {totals['top1']/n:.3f} | top5 {totals['top5']/n:.3f} | "
        f"value_mean {totals['value_sum']/n:+.3f} target_mean {totals['target_sum']/n:+.3f}"
    )

In [ ]:
# --- train on engine value targets ---------------------------------------
from chess_gnn.train_sl import train

CKPT_DIR = REPO_DIR / "checkpoints" / "sl_stockfish"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

train(
    pgn_paths=[PGN_PATH],
    ckpt_dir=CKPT_DIR,
    steps=25_000,
    batch_size=128,
    lr=3e-4,
    hidden_dim=128,
    num_layers=4,
    num_heads=4,
    min_elo=MIN_ELO,
    num_workers=2,
    log_every=100,
    ckpt_every=2500,
    device=DEVICE,
    value_coef=0.75,
    engine_value_path=TRAIN_VALUE_DATA,
    engine_value_blend=1.0,
)
evaluate_sl_checkpoint(CKPT_DIR / "sl_final.pt")

In [ ]:
# --- inspect predictions -------------------------------------------------
import chess
from IPython.display import SVG, display

from chess_gnn.play import GNNAgent
from chess_gnn.viz import render_prediction_svg

model = load_model(CKPT_DIR / "sl_final.pt", device=DEVICE)
agent = GNNAgent(model, device=DEVICE, default_temperature=0.3, num_simulations=0)

board = chess.Board()
board.push_san("e4"); board.push_san("e5"); board.push_san("Nf3")
ranking = agent.rank_moves(board)
for m, p in list(zip(ranking.moves, ranking.probabilities))[:8]:
    print(f"  {m.uci():>5s}  {p:.3f}")
display(SVG(render_prediction_svg(board, ranking, topk=6)))

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/chess/checkpoints /content/drive/MyDrive/chess-gnn-checkpoints

